In [13]:
import json
from src.RelationE.sentences import sentence_spans, assign_sentence_ids
from src.RelationE.models import Mention
from src.configs.re_cfg import CandidateConfig

from dataclasses import dataclass
import argparse
from itertools import combinations
from collections import Counter

from src.RelationE.models import Mention, CandidatePair, ContextGroup
from src.RelationE.sentences import context_boundaries, sentence_distance, character_distance
from src.RelationE.schema import compatible_relations, FINE_RELATIONS, RelationDefinition, get_schema

from src.util.names import is_same_entity
from src.configs.re_cfg import CandidateConfig
from src.RelationE.generator import generate_candidates

In [8]:
with open('data/entity_annotations_raw.jsonl', 'r', encoding='utf-8') as f:
    lines = [json.loads(line) for line in f.readlines()]


In [ ]:

ids = 0
text = lines[ids]['text']
entities = lines[ids]['entities']

sent_spans = sentence_spans(text)

mentions = [Mention(i, ent['text'], ent['type'], ent['start'], ent['end']) for i, ent in enumerate(entities)]
assign_sentence_ids(mentions, sent_spans)

# parser = argparse.ArgumentParser()
cfg = CandidateConfig()

candidates = generate_candidates(
    text=text,
    mentions=mentions,
    spans=sent_spans,
    schema=FINE_RELATIONS,
    stats=None,
    cfg=cfg
)


for i, c in enumerate(candidates):
    # if i>10:
    #     break
    print("\n", c, "\n")
    print(text[c.context_start:c.context_end], "\n")
    break



 CandidatePair(pair_id='P00001', entity1=Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3), entity2=Mention(mention_id=1, text='Iagerhuis', entity_type='Organization', start=134, end=143, normalized_name=None, confidence=None, sentence_id=3), candidate_relations=['governs', 'controls', 'attacked', 'defeated', 'deployed_in', 'traveled_to', 'departed_from', 'located_in', 'based_in'], context_start=16, context_end=381, sentence_distance=0, character_distance=65) 

uitzending van 17 Juli 1939. Copie BUITENIAND. LONDEN: In antwoord op een vraag van een arbeidersafgevaardigde in het Iagerhuis heeft minister CHAMBERIAIN verklaard, dat het hem niet mogelijk was de verzekering te geven, dat het Huis niet op reces zou gaan voor de onderhandelingen met Rusland geeindigd zouden zijn. Het is waarschijlijk dat vanavond de opmerkingen 



In [49]:
candidates[2].candidate_relations
mentions

[Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=1, text='Iagerhuis', entity_type='Organization', start=134, end=143, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=2, text='CHAMBERIAIN', entity_type='Person', start=159, end=170, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=3, text='onderhandelingen met Rusland', entity_type='Event', start=281, end=309, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=4, text='Rusland', entity_type='Place', start=302, end=309, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=5, text='engelsche regeering', entity_type='Organization', start=389, end=408, normalized_name=None, confidence=None, sentence_id=4),
 Mention(mention_id=6, text='TOKIO', entity_type='Place', start=459, end=464, normalized_name=None, confidence=None, sentence_id=4),
 Menti

In [31]:
from src.RelationE.jev import render_marked_context, build_relation_options

In [51]:
cmap, coptions = build_relation_options(mentions[1], mentions[2], candidates[2].candidate_relations)
render_marked_context(text, mentions, 0, 100)

AttributeError: 'int' object has no attribute 'replace'

In [36]:
# render_marked_context(text=text, mentions=mentions, start=63, end=332)

In [19]:
mentions

[Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=1, text='Iagerhuis', entity_type='Organization', start=134, end=143, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=2, text='CHAMBERIAIN', entity_type='Person', start=159, end=170, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=3, text='onderhandelingen met Rusland', entity_type='Event', start=281, end=309, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=4, text='Rusland', entity_type='Place', start=302, end=309, normalized_name=None, confidence=None, sentence_id=3),
 Mention(mention_id=5, text='engelsche regeering', entity_type='Organization', start=389, end=408, normalized_name=None, confidence=None, sentence_id=4),
 Mention(mention_id=6, text='TOKIO', entity_type='Place', start=459, end=464, normalized_name=None, confidence=None, sentence_id=4),
 Menti

In [29]:
import html

def render_marked_context_(
    text: str,
    mentions,
    start: int,
    end: int,
) -> str:
    """
    Render a text context with entity markers.

    The surrounding context remains ordinary text.

    Example:

        The <e id="M1" type="Person">Chamberlain</e>
        met with <e id="M2" type="Person">Ribbentrop</e>.

    Original text is HTML/XML escaped where necessary.
    This prevents OCR characters such as '<' and '&' from
    accidentally breaking the marker structure.
    """

    relevant = [
        mention
        for mention in mentions
        if mention.start >= start
        and mention.end <= end
    ]

    relevant.sort(
        key=lambda mention: (
            mention.start,
            mention.end,
        )
    )

    parts = []
    cursor = start

    for mention in relevant:

        if mention.start < cursor:
            raise ValueError(
                "Overlapping entity mentions cannot be represented "
                f"with simple markers: {mention.mention_id}"
            )

        # Preserve the surrounding text.
        parts.append(
            html.escape(text[cursor:mention.start], quote=False)
        )

        # Mark the entity mention.
        mention_id = html.escape(
            mention.mention_id,
            quote=True,
        )

        entity_type = html.escape(
            mention.entity_type,
            quote=True,
        )

        mention_text = html.escape(
            text[mention.start:mention.end],
            quote=False,
        )

        parts.append(
            f'<e id="{mention_id}" type="{entity_type}">'
            f'{mention_text}'
            f'</e>'
        )

        cursor = mention.end

    parts.append(
        html.escape(text[cursor:end], quote=False)
    )

    return "".join(parts)


In [30]:
render_marked_context_(text=text, mentions=mentions, start=63, end=332)

AttributeError: 'int' object has no attribute 'replace'

In [65]:

from src.RelationE.io import load_bulletins
from src.RelationE.generator import generate_candidates, group_candidates

In [59]:
bulletins = list(load_bulletins('data/entity_annotations_raw.jsonl'))

In [66]:
candidates = generate_candidates(mentions, text, sent_spans, FINE_RELATIONS)
context_groups = group_candidates(candidates)
context_groups

[ContextGroup(context_id='C00001', context_start=16, context_end=381, pairs=[CandidatePair(pair_id='P00001', entity1=Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3), entity2=Mention(mention_id=1, text='Iagerhuis', entity_type='Organization', start=134, end=143, normalized_name=None, confidence=None, sentence_id=3), candidate_relations=['governs', 'controls', 'attacked', 'defeated', 'deployed_in', 'traveled_to', 'departed_from', 'located_in', 'based_in'], context_start=16, context_end=381, sentence_distance=0, character_distance=65), CandidatePair(pair_id='P00002', entity1=Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3), entity2=Mention(mention_id=2, text='CHAMBERIAIN', entity_type='Person', start=159, end=170, normalized_name=None, confidence=None, sentence_id=3), candidate_relations=['governs', 'attacked', 'defeated', 'trav

In [69]:
context_groups[0].pairs
context_groups

[ContextGroup(context_id='C00001', context_start=16, context_end=381, pairs=[CandidatePair(pair_id='P00001', entity1=Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3), entity2=Mention(mention_id=1, text='Iagerhuis', entity_type='Organization', start=134, end=143, normalized_name=None, confidence=None, sentence_id=3), candidate_relations=['governs', 'controls', 'attacked', 'defeated', 'deployed_in', 'traveled_to', 'departed_from', 'located_in', 'based_in'], context_start=16, context_end=381, sentence_distance=0, character_distance=65), CandidatePair(pair_id='P00002', entity1=Mention(mention_id=0, text='LONDEN', entity_type='Place', start=63, end=69, normalized_name=None, confidence=None, sentence_id=3), entity2=Mention(mention_id=2, text='CHAMBERIAIN', entity_type='Person', start=159, end=170, normalized_name=None, confidence=None, sentence_id=3), candidate_relations=['governs', 'attacked', 'defeated', 'trav

In [78]:
ids = 3
render_marked_context(
    bulletins[0].text, 
    bulletins[0].mentions, 
    context_groups[ids].context_start, 
    context_groups[ids].context_end)

'bliksem getroffen en brandend neergestort. <e id="e15" type="Place">JERUZAIEM</e id="e15"> Uit protest tegen de voorgestelde schorsing van de joodsche im 5 migratie heeft de <e id="e16" type="Organization">joodsche nationale raad</e id="e16"> <e id="e17" type="Event">een staking van tien uur</e id="e17"> geproclameerd, Van morgen 14 uur tot middernacht zal <e id="e18" type="Event">deze staking</e id="e18"> van arbeid, handel en verkeer duren.'

In [79]:
from src.RelationE.jev import build_jev_batches

In [81]:
batch = build_jev_batches(bulletins[0], context_groups, FINE_RELATIONS, 30, choice_factory = None, shared_instructions = True)

In [83]:
for b in batch:
    print(b)

JEVBatch(state={'bulletin_id': 'anp:anp:1939:07:17:4:mpeg21', 'bulletin_date': '1939-07-17', 'contexts': [{'context_id': 'C00001', 'context': 'uitzending van 17 Juli 1939. Copie BUITENIAND. <e id="e1" type="Place">LONDEN</e id="e1">: In antwoord op een vraag van een arbeidersafgevaardigde in het <e id="e2" type="Organization">Iagerhuis</e id="e2"> heeft minister <e id="e3" type="Person">CHAMBERIAIN</e id="e3"> verklaard, dat het hem niet mogelijk was de verzekering te geven, dat het Huis niet op reces zou gaan voor de <e id="e4" type="Event">onderhandelingen met <e id="e5" type="Place">Rusland</e id="e5"></e id="e4"> geeindigd zouden zijn. Het is waarschijlijk dat vanavond de opmerkingen', 'target_pairs': [{'pair_id': 'P00001', 'entity1': {'mention_id': 0, 'type': 'Place'}, 'entity2': {'mention_id': 1, 'type': 'Organization'}}, {'pair_id': 'P00002', 'entity1': {'mention_id': 0, 'type': 'Place'}, 'entity2': {'mention_id': 2, 'type': 'Person'}}, {'pair_id': 'P00003', 'entity1': {'mention

In [89]:
batch[0].state['instructions'] #instructions, contexts

'You are a relation classification system for historical Dutch radio bulletins from 1939.\n\nClassify the relationship between two specified entity mentions in a bulletin.\n\nINPUT\nThe shared state contains contexts with original Dutch text (OCR: old spelling, errors). Entity mentions are marked <e id="..." type="...">text</e id="...">; the closing tag repeats\nthe id because mentions can be nested, e.g. a match phrase containing two clubs. Each question names one target pair and one context.\n\nRULES\n1. Classify only the target pair of the question.\n2. Mention ids identify occurrences, not necessarily different real-world entities. Types come from the NER system: do not reclassify.\n3. Select exactly one of the provided options. A relation must be supported by the supplied text; do not use outside historical knowledge.\n4. The order in the text does not decide the direction. For a directional relation choose the option with the correct head (subject) and tail (object); for a symmet

In [87]:
batch[0].questions

{'q_P00001': Choice(type='choice', instructions='CLASSIFICATION TASK: use context C00001. Classify pair P00001: entity 1 = 0 (type: Place), entity 2 = 1 (type: Organization). Select exactly one option.', criteria={'governs__entity2_to_entity1': "The subject holds political or executive authority over the object Place. Examples include a prime minister governing a country or a mayor governing a city. Look for explicit evidence such as 'burgemeester van [Place]' or 'regeert over [Place]'. A title alone is not sufficient. Semantic direction: 1 is the head (subject) and 0 is the tail (object).", 'controls__entity2_to_entity1': "The subject Organization has operational, political or military control over the object Organization or Place. Look for explicit evidence such as 'beheerst', 'onder gezag van' or 'bezette [Place]'. Semantic direction: 1 is the head (subject) and 0 is the tail (object).", 'attacked__entity2_to_entity1': "The subject Person, Organization or Vehicle attacked the object